<a href="https://colab.research.google.com/github/Yashika-Bayeen/agentic-ai/blob/main/Day_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install pydantic google-genai groq python-dotenv -q
print("✅ Libraries installed.")

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Define the schema
class UserProfile(BaseModel):
    name: str = Field(description="The user's full name.")
    age: int = Field(description="The age in years.")
    skills: List[str] = Field(description="List of programming skills.")
    location: Optional[str] = Field(None, description="City and country of residence.")

# Note: The code below illustrates how we would call Gemini 2.5 Flash's response_schema in the official SDK:
"""
from google import genai
client = genai.Client()
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Create a profile for Alice, 25 years old from Berlin, who knows Python and Go.',
    config=dict(
        response_mime_type="application/json",
        response_schema=UserProfile,
    ),
)
print(response.text)
"""
print("✅ Schema defined and structured output pattern loaded.")

In [ ]:
# A mock database of employees
DB_EMPLOYEES = [
    {'id': 1, 'name': 'Priya Sharma', 'department': 'Engineering', 'role': 'Senior Developer', 'salary': 1800000},
    {'id': 2, 'name': 'Aisha Khan', 'department': 'Data', 'role': 'Data Scientist', 'salary': 1600000},
    {'id': 3, 'name': 'Neha Gupta', 'department': 'Design', 'role': 'UX Designer', 'salary': 1100000},
]

def get_employees(department: str) -> List[dict]:
    """Fetch employees working in a given department."""
    return [emp for emp in DB_EMPLOYEES if emp['department'].lower() == department.lower()]

# JSON Schema description we feed to the LLM so it knows get_employees exists:
tool_schema = {
    "name": "get_employees",
    "description": "Fetch employee list filtering by department name.",
    "parameters": {
        "type": "object",
        "properties": {
            "department": {
                "type": "string",
                "description": "The department name (e.g. Engineering, Data, Design)"
            }
        },
        "required": ["department"]
    }
}
print("✅ Tool and its JSON schema defined.")

In [ ]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

messages = [
    {"role": "user", "content": "Who works in the Data department?"}
]

tools = [{
    "type": "function",
    "function": tool_schema
}]

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

response_message = response.choices[0].message
tool_calls = response_message.tool_calls

if tool_calls:
    print("🤖 LLM requested tool calling:")
    for call in tool_calls:
        print(f"  Function: {call.function.name}")
        print(f"  Arguments: {call.function.arguments}")
else:
    print("🤖 LLM answered text directly:", response_message.content)

In [ ]:
def add_new_employee(name: str, department: str, role: str, salary: int) -> str:
    """Add a new employee record to the database."""
    new_id = len(DB_EMPLOYEES) + 1
    new_emp = {'id': new_id, 'name': name, 'department': department, 'role': role, 'salary': salary}
    DB_EMPLOYEES.append(new_emp)
    return f"Success: Created employee {name} with ID {new_id}."

def update_employee_salary(emp_id: int, new_salary: int) -> str:
    """Update the salary of an existing employee by ID."""
    for emp in DB_EMPLOYEES:
        if emp['id'] == emp_id:
            old_sal = emp['salary']
            emp['salary'] = new_salary
            return f"Success: Updated employee ID {emp_id} salary from {old_sal} to {new_salary}."
    return f"Error: Employee ID {emp_id} not found."

print("✅ Write and update tools defined.")

In [ ]:
def validate_tool_args(func_name, arguments):
    if func_name == "add_new_employee":
        if arguments.get("salary", 0) <= 0:
            raise ValueError("Salary must be a positive integer.")
    if func_name == "update_employee_salary":
        if arguments.get("new_salary", 0) <= 0:
            raise ValueError("New salary must be a positive integer.")

    print(f"Validation passed for {func_name}")
    return True

# Test validation guard
try:
    validate_tool_args("add_new_employee", {"name": "Bob", "salary": -5000})
except ValueError as e:
    print("Guard caught error:", e)

In [ ]:
import json

# Define schemas for LLM registration
write_schemas = [
    {
        "name": "add_new_employee",
        "description": "Add a new employee to the database.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "department": {"type": "string"},
                "role": {"type": "string"},
                "salary": {"type": "integer"}
            },
            "required": ["name", "department", "role", "salary"]
        }
    },
    {
        "name": "update_employee_salary",
        "description": "Update salary of an existing employee by ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "emp_id": {"type": "integer"},
                "new_salary": {"type": "integer"}
            },
            "required": ["emp_id", "new_salary"]
        }
    }
]

available_tools = {
    "get_employees": get_employees,
    "add_new_employee": add_new_employee,
    "update_employee_salary": update_employee_salary
}

all_tool_schemas = [tool_schema] + write_schemas

def run_agentic_loop(user_query):
    print(f"User: {user_query}")
    messages = [{"role": "user", "content": user_query}]
    formatted_tools = [{"type": "function", "function": s} for s in all_tool_schemas]

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        tools=formatted_tools
    )

    resp_msg = response.choices[0].message
    messages.append(resp_msg)

    if resp_msg.tool_calls:
        for tool_call in resp_msg.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)

            # Pre-validation guard
            try:
                validate_tool_args(func_name, func_args)
            except ValueError as ve:
                print(f"Blocked function call: {ve}")
                return f"Execution stopped due to validation failure: {ve}"

            # Execute
            print(f"Executing tool '{func_name}' with args {func_args}...")
            func_to_call = available_tools[func_name]
            tool_result = func_to_call(**func_args)
            print(f"Tool response: {tool_result}")

            # Feed tool result back to LLM
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result)
            })

        # Final conversational reply
        final_response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages
        )
        return final_response.choices[0].message.content
    else:
        return resp_msg.content

# Run write query
print("Agent Reply:", run_agentic_loop("Please add a new employee named Bob Jones in Design as a Graphic Designer with a salary of 900000"))
print("-----------------------------------")
# Run list query
print("Agent Reply:", run_agentic_loop("Who is currently working in Design?"))